Lab 2 Assignment
HESVIKA SOMUNAIDU
IS01084471

In [27]:
import pandas as pd
import numpy as np

# Text processing
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Feature extraction
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Evaluation
from sklearn.metrics import classification_report, accuracy_score

# Lexicon-based
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# VADER lexicon
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/4e4d763e-33da-474c-88ed-
[nltk_data]     bac8348e6ba4/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [2]:
df = pd.read_csv("Reviews.csv", nrows=50000)

# Keep only needed columns
df = df[['Score', 'Text']]

# Convert score to sentiment
def label_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Sentiment'] = df['Score'].apply(label_sentiment)

df.head()

,Score,Text,Sentiment
0,5,I have bought several of the Vitality canned d...,positive
1,1,Product arrived labeled as Jumbo Salted Peanut...,negative
2,4,This is a confection that has been around a fe...,positive
3,2,If you are looking for the secret ingredient i...,negative
4,5,Great taffy at a great price. There was a wid...,positive


In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# Download once
nltk.download('stopwords')

# Stopwords
stop_words = set(stopwords.words('english'))
more_stopwords = {'u', 'im', 'c'}
stop_words = stop_words.union(more_stopwords)

# Stemmer
stemmer = SnowballStemmer("english")

def preprocess(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\b\w*\d\w*\b', '', text)

    words = text.split()
    
    # Remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # Stemming
    words = [stemmer.stem(word) for word in words]

    return " ".join(words)

# Apply
df['Clean_Text'] = df['Text'].apply(preprocess)

# Preview
print(df[['Text', 'Clean_Text']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/4e4d763e-33da-474c-88ed-
[nltk_data]     bac8348e6ba4/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                                Text  \
0  I have bought several of the Vitality canned d...   
1  Product arrived labeled as Jumbo Salted Peanut...   
2  This is a confection that has been around a fe...   
3  If you are looking for the secret ingredient i...   
4  Great taffy at a great price.  There was a wid...   

                                          Clean_Text  
0  bought sever vital can dog food product found ...  
1  product arriv label jumbo salt peanutsth peanu...  
2  confect around centuri light pillowi citrus ge...  
3  look secret ingredi robitussin believ found go...  
4  great taffi great price wide assort yummi taff...  


In [4]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['Clean_Text'])
y = df['Sentiment']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Lexicon-Based Method (TextBlob)

In [9]:
def get_textblob_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    
    if polarity > 0:
        return 'positive'
    elif polarity < 0:
        return 'negative'
    else:
        return 'neutral'

df['TextBlob_Sentiment'] = df['Text'].apply(get_textblob_sentiment)

print(classification_report(df['Sentiment'], df['TextBlob_Sentiment']))

              precision    recall  f1-score   support

    negative       0.54      0.36      0.43      7535
     neutral       0.08      0.01      0.02      4047
    positive       0.82      0.94      0.88     38418

    accuracy                           0.78     50000
   macro avg       0.48      0.44      0.44     50000
weighted avg       0.72      0.78      0.74     50000



Lexicon-Based Method (VADER)

In [13]:
sia = SentimentIntensityAnalyzer()

In [14]:
def get_vader_sentiment(text):
    if not isinstance(text, str):
        return "neutral"
    
    score = sia.polarity_scores(text)
    compound = score['compound']
    
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

In [15]:
df['VADER_Sentiment'] = df['Text'].apply(get_vader_sentiment)

In [16]:
print("VADER Accuracy:", accuracy_score(df['Sentiment'], df['VADER_Sentiment']))
print(classification_report(df['Sentiment'], df['VADER_Sentiment']))

VADER Accuracy: 0.79066
              precision    recall  f1-score   support

    negative       0.57      0.40      0.47      7535
     neutral       0.15      0.04      0.06      4047
    positive       0.83      0.95      0.89     38418

    accuracy                           0.79     50000
   macro avg       0.52      0.46      0.47     50000
weighted avg       0.74      0.79      0.76     50000



Machine Learning Model (Naive Bayes)

In [17]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.7936
              precision    recall  f1-score   support

    negative       0.84      0.23      0.36      1545
     neutral       0.50      0.00      0.01       838
    positive       0.79      0.99      0.88      7617

    accuracy                           0.79     10000
   macro avg       0.71      0.41      0.42     10000
weighted avg       0.77      0.79      0.73     10000



Machine Learning Model (SVM)

In [20]:
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.8383
              precision    recall  f1-score   support

    negative       0.70      0.63      0.67      1545
     neutral       0.41      0.13      0.20       838
    positive       0.88      0.96      0.91      7617

    accuracy                           0.84     10000
   macro avg       0.66      0.58      0.59     10000
weighted avg       0.81      0.84      0.82     10000



In [22]:
print("Model Comparison:")

# Lexicon-Based Models (use full dataset OR same subset)
print("VADER Accuracy:", accuracy_score(df['Sentiment'], df['VADER_Sentiment']))
print("TextBlob Accuracy:", accuracy_score(df['Sentiment'], df['TextBlob_Sentiment']))

# Machine Learning Models (use test set)
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

Model Comparison:
VADER Accuracy: 0.79066
TextBlob Accuracy: 0.77934
Naive Bayes Accuracy: 0.7936
SVM Accuracy: 0.8383


Based on the results, SVM achieved the highest accuracy (0.8383), followed by Naive Bayes (0.7936), VADER (0.79066) and TextBlob (0.77934). This shows that Machine Learning models generally perform better than Lexicon-based methods.

SVM works well because it can handle complex text data and find better patterns, but it takes more time to train. Naive Bayes is faster and simpler, but its assumption that words are independent can reduce accuracy.

VADER performs quite well even without training and is good at handling things like punctuation and emojis. However, it cannot learn from the data, so its performance is limited. TextBlob is the easiest to use, but it is less accurate because it does not consider context or emphasis.

Overall, SVM is the best model for accuracy, while VADER and TextBlob are useful for quick and simple sentiment analysis.

In [31]:
df[['Score', 'Sentiment', 'Clean_Text']].to_csv('Extracted_Reviews2.csv', index=False)
print("\nData exported to Extracted_Reviews2.csv successfully!")


Data exported to Extracted_Reviews2.csv successfully!
